In [43]:
import sympy as sym
from sympy import *
#Coefficient for LHQ (cluster 1) from ASICS
A = [-1.730215,-2.744504,-3.396561]
B = [1.251296,2.561384,-1.133846]
C = [3.573296,0.986618,-0.473004]
D = [-0.060956,-0.376258,-0.980912]
E = [-0.037878,-0.579924,-1.166911]

Ispin = 3/2
w0 = 192.55 #Larmor Frequency for 11B 

wkhz = w0*10**3 #Larmor freq in kHz

#factor q given in article
q = (3-4*Ispin*(Ispin + 1))/(16*wkhz)

#Define symbol and force them to be real
AzzmAyy,AzzmAxx,AyymAxx,Ayz,Axz,Axy = sym.symbols('AzzmAyy,AzzmAxx,AyymAxx,Ayz,Axz,Axy', real=True)

#Variable for each equation set
quad_tensor = [(AzzmAyy, Ayz), (AzzmAxx, Axz), (AyymAxx, Axy)]

# List to hold solutions
solutions = []

for i, (diagonal_diff, off_diagonal) in enumerate(quad_tensor):
    eq1 = sym.Eq(-((diagonal_diff)**2 - 4*(off_diagonal)**2)*((9*q/8)), D[i])
    eq2 = sym.Eq(off_diagonal*(diagonal_diff)*(9*q/2), E[i])

    # Solve the system
    solution = sym.solve([eq1, eq2], (diagonal_diff, off_diagonal))
    solutions.append(solution)

# Assign the solutions to the respective variables
AzzmAyy = [solutions[0][0][0], solutions[0][1][0]]
Ayz = [solutions[0][0][1], solutions[0][1][1]]
AzzmAxx = [solutions[1][0][0], solutions[1][1][0]]
Axz = [solutions[1][0][1], solutions[1][1][1]]
AyymAxx = [solutions[2][0][0], solutions[2][1][0]]
Axy = [solutions[2][0][1], solutions[2][1][1]]

# Print the final results for the variables
print(f"Azz - Ayy: {AzzmAyy}, Ayz: {Ayz}")
print(f"Azz - Axx: {AzzmAxx}, Axz: {Axz}")
print(f"Ayy - Axx: {AyymAxx}, Axy: {Axy}")  



Azz - Ayy: [-35.1208684028640, 35.1208684028640], Ayz: [-61.5306552120510, 61.5306552120510]
Azz - Axx: [-189.595156285395, 189.595156285395], Axz: [-174.507296397013, 174.507296397013]
Ayy - Axx: [-249.031671571717, 249.031671571717], Axy: [-267.333199332134, 267.333199332134]


In [44]:
import itertools
# Generate all combinations of AzzmAxx, AyymAxx, and AzzmAyy
combinations = list(itertools.product(AzzmAxx, AyymAxx, AzzmAyy))

In [45]:
import numpy as np
#Find Quadrupolar tensor diagonal elements

Axx1 = []; Axx2 = []; Axx3 = []
Ayy1 = []; Ayy2 = []; Ayy3 = []
Azz1 = []; Azz2 = []; Azz3 = []

# Initialize variables to track the best combination and minimum variation
best_combination = None
min_variation = float('inf')
for (AzzmAxx_val, AyymAxx_val, AzzmAyy_val) in combinations:
    # Solution 1
    Axx1_val = (-(AzzmAxx_val + AyymAxx_val)/3)
    Ayy1_val = Axx1_val + AyymAxx_val
    Azz1_val = Axx1_val + AzzmAxx_val

    #Save values
    Axx1.append(Axx1_val)
    Ayy1.append(Ayy1_val)
    Azz1.append(Azz1_val)

     # Solution 2
    Ayy2_val = -(AzzmAyy_val - AyymAxx_val) / 3
    Axx2_val = Ayy2_val - AyymAxx_val
    Azz2_val = Ayy2_val + AzzmAyy_val

     #Save values
    Axx2.append(Axx2_val)
    Ayy2.append(Ayy2_val)
    Azz2.append(Azz2_val)

    # Solution 3
    Azz3_val = (AzzmAxx_val + AzzmAyy_val) / 3
    Axx3_val = Azz3_val - AzzmAxx_val
    Ayy3_val = Azz3_val - AzzmAyy_val
    
    #Save values
    Axx3.append(Axx3_val)
    Ayy3.append(Ayy3_val)
    Azz3.append(Azz3_val)

    # Convert sympy Float to regular Python float for NumPy functions
    Axx1_val = float(Axx1_val)
    Axx2_val = float(Axx2_val)
    Axx3_val = float(Axx3_val)
    
    Ayy1_val = float(Ayy1_val)
    Ayy2_val = float(Ayy2_val)
    Ayy3_val = float(Ayy3_val)
    
    Azz1_val = float(Azz1_val)
    Azz2_val = float(Azz2_val)
    Azz3_val = float(Azz3_val)

    # Calculate variation (standard deviation) for Axx, Ayy, Azz
    variation_Axx = np.std([Axx1_val, Axx2_val, Axx3_val])
    variation_Ayy = np.std([Ayy1_val, Ayy2_val, Ayy3_val])
    variation_Azz = np.std([Azz1_val, Azz2_val, Azz3_val])

    total_variation = variation_Axx + variation_Ayy + variation_Azz

    # Update the best combination if the current one has less variation
    if total_variation < min_variation:
        min_variation = total_variation
        best_combination = (AzzmAxx_val, AyymAxx_val, AzzmAyy_val)
        best_avg_Axx = np.mean([Axx1_val, Axx2_val, Axx3_val])
        best_avg_Ayy = np.mean([Ayy1_val, Ayy2_val, Ayy3_val])
        best_avg_Azz = np.mean([Azz1_val, Azz2_val, Azz3_val])

#Get index for off-diagonal elements        
index_AzzmAxx = AzzmAxx.index(best_combination[0])
best_Axz = Axz[index_AzzmAxx]

index_AyymAxx = AyymAxx.index(best_combination[1])
best_Axy = Axy[index_AyymAxx]

index_AzzmAyy = AzzmAyy.index(best_combination[2])
best_Ayz = Ayz[index_AzzmAyy]

# Print results
print("Axx1:", Axx1)
print("Axx2:", Axx2)
print("Axx3:", Axx3)

print("Ayy1:", Ayy1)
print("Ayy2:", Ayy2)
print("Ayy3:", Ayy3)

print("Azz1:", Azz1)
print("Azz2:", Azz2)
print("Azz3:", Azz3)

print("Best combination with minimum standard deviation:")
print("AzzmAxx:", best_combination[0])
print("AyymAxx:", best_combination[1])
print("AzzmAyy:", best_combination[2])
print("Axz:", best_Axz)
print("Axy:", best_Axy)
print("Ayz:", best_Ayz)

print("Average of Axx1, Axx2, Axx3 with minimum standard deviation:", best_avg_Axx)
print("Average of Ayy1, Ayy2, Ayy3 with minimum standard deviation:", best_avg_Ayy)
print("Average of Azz1, Azz2, Azz3 with minimum standard deviation:", best_avg_Azz)


Axx1: [146.208942619037, 146.208942619037, -19.8121717621073, -19.8121717621073, 19.8121717621073, 19.8121717621073, -146.208942619037, -146.208942619037]
Axx2: [177.728070515432, 154.314158246856, -154.314158246856, -177.728070515432, 177.728070515432, 154.314158246856, -154.314158246856, -177.728070515432]
Axx3: [114.689814722642, 138.103726991218, 114.689814722642, 138.103726991218, -138.103726991218, -114.689814722642, -138.103726991218, -114.689814722642]
Ayy1: [-102.822728952680, -102.822728952680, 229.219499809609, 229.219499809609, -229.219499809609, -229.219499809609, 102.822728952680, 102.822728952680]
Ayy2: [-71.3036010562842, -94.7175133248602, 94.7175133248602, 71.3036010562842, -71.3036010562842, -94.7175133248602, 94.7175133248602, 71.3036010562842]
Ayy3: [-39.7844731598888, -86.6122976970408, -39.7844731598888, -86.6122976970408, 86.6122976970408, 39.7844731598888, 86.6122976970408, 39.7844731598888]
Azz1: [-43.3862136663575, -43.3862136663575, -209.407328047502, -209.4